# FanDuel quarterly scorecard

Compare the same calendar months in Massachusetts sportsbook, Michigan sportsbook and Michigan casino. Amounts remain **native USD**; shares and gross hold are percentages, changes are percentage points. Each state and product remains separate.

This is a diagnostic view of a current capture, not an approved forecast or a reconstruction of what was known historically. A July-only Q3 window is one month out of three. Future refreshes can revise old observations; conflicting versions require review.


In [ ]:
database_file = "data/staging/refresh_20260912T201854Z/gaming_current.sqlite"
quarter = None       # e.g. "2026Q3"; None uses the latest supported captured month
through_month = None # e.g. "2026-07"; explicit same-window comparison, not extrapolation


In [ ]:
from pathlib import Path
import hashlib
import json
import sqlite3
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from variant_gaming.flut_scorecard import (
    CONTRACTS, build_monthly_scorecard, build_quarterly_scorecard,
    scorecard_scope, sportsbook_hold,
)

DB = ROOT / database_file
manifest = json.loads((DB.parent / "run_manifest.json").read_text())
validation = json.loads((DB.parent / "validation.json").read_text())
if not manifest.get("finished_at"):
    raise RuntimeError("Capture is unfinished; inspect its collection log.")
before = hashlib.sha256(DB.read_bytes()).hexdigest()
if before != validation["database_sha256"]:
    raise RuntimeError("Database differs from its validated capture.")
with sqlite3.connect(DB.resolve().as_uri() + "?mode=ro", uri=True) as connection:
    observations = pd.read_sql_query("SELECT * FROM gaming_results", connection)
    coverage = pd.read_sql_query("SELECT * FROM source_coverage", connection)
print("Capture:", manifest["finished_at"], "| Research status: diagnostic / unreviewed")
print("Database:", DB)


## Source definitions and coverage

Michigan's older database label `Adjusted Gross` describes only one workbook column: the parser separately retained **Gross Receipts** and **Adjusted Gross**. The table below names each actual source metric. Massachusetts gross revenue is **Accrual Win**; its taxable revenue remains separate.

Printed statewide controls must pass the exact source contract and reconcile against all retained operators. Amounts must come from one common source version. Equal copies retain all references; conflicting values cannot be selected by capture date.


In [ ]:
definitions = pd.DataFrame([
    {"state": state, "product": vertical, "native_FanDuel_identity": contract["operator"],
     "stored_column": metric, "source_metric": label}
    for (state, vertical), contract in CONTRACTS.items()
    for metric, label in contract["metrics"].items()
])
display(definitions)
scope = scorecard_scope(observations)
display(scope)
# Coverage-only registrations can have no observation rows at all.
display(coverage[["state_code", "vertical", "status", "reason", "latest_period"]])


In [ ]:
monthly = build_monthly_scorecard(observations)
if monthly.empty:
    raise RuntimeError("No supported monthly evidence is present in this capture.")
source_refs = sorted({ref for refs in monthly.source_refs for ref in refs})
source_table = pd.DataFrame(source_refs, columns=["source_url", "source_file", "source_sha256"])
for source_file, expected_hash in source_table[["source_file", "source_sha256"]].itertuples(index=False, name=None):
    raw_path = (ROOT / source_file).resolve()
    if not raw_path.is_relative_to((ROOT / "data" / "raw").resolve()):
        raise RuntimeError("Source path leaves the retained raw directory.")
    if hashlib.sha256(raw_path.read_bytes()).hexdigest() != expected_hash:
        raise RuntimeError(f"Source bytes differ from their captured hash: {source_file}")
print("Verified source references:", len(source_table))
display(monthly[monthly.status.ne("eligible")][[
    "state_code", "vertical", "period_start", "metric", "status", "reason"]])


## Same-window quarter comparison

Every requested month must qualify in both years. Missing or excluded months block the comparison; they are never filled with zero or silently removed. Market share is the ratio of summed FanDuel amounts to summed statewide amounts. Percentage growth is undefined when the prior FanDuel amount is zero or negative.


In [ ]:
capture_day = pd.Timestamp(manifest["finished_at"]).tz_convert("UTC").tz_localize(None).normalize()
if pd.to_datetime(monthly.period_end).max() > capture_day:
    raise RuntimeError("A supported period ends after this capture; investigate before using it.")
latest_supported_month = pd.to_datetime(monthly.period_end).max().to_period("M")
selected_through = pd.Period(through_month or str(latest_supported_month), freq="M")
selected_quarter = quarter or str(selected_through.asfreq("Q-DEC"))
quarterly = build_quarterly_scorecard(monthly, quarter=selected_quarter, through_month=str(selected_through))
print("Requested window:", selected_quarter, "through", selected_through)
display(quarterly[["state_code", "vertical", "native_metric", "expected_months",
    "observed_window_months", "expected_window_months", "matched_window_months", "quarter_months",
    "quarter_complete", "status", "missing_or_excluded_months"]])


In [ ]:
display(quarterly[["state_code", "vertical", "native_metric", "fd_amount", "prior_fd_amount",
    "fd_growth_pct", "market_amount", "prior_market_amount", "market_growth_pct",
    "fd_share_pct", "prior_fd_share_pct", "share_change_pp", "growth_status", "market_growth_status", "share_status"]].round(3))
display(sportsbook_hold(quarterly).round(3))
print("Gross hold is source-native gross revenue / handle. Casino has no handle or sportsbook hold.")


## Visible monthly inputs and sources

These are the current and prior-year months used above. Negative revenue remains negative. A source can reconcile while its market revenue is nonpositive; in that case the native amounts remain visible and the share is undefined.


In [ ]:
q = pd.Period(selected_quarter, freq="Q-DEC")
current_months = pd.period_range(q.start_time, selected_through.start_time, freq="M")
input_months = {str(month) for month in current_months} | {str(month-12) for month in current_months}
window_inputs = monthly[monthly.period_start.str[:7].isin(input_months)]
display(window_inputs[["state_code", "vertical", "period_start", "native_metric", "fd_amount",
    "market_amount", "operator_sum", "reconciliation_difference", "status", "reason"]])
window_refs = sorted({ref for refs in window_inputs.source_refs for ref in refs})
display(pd.DataFrame(window_refs, columns=["source_url", "source_file", "source_sha256"]))
assert hashlib.sha256(DB.read_bytes()).hexdigest() == before
print("Read-only review complete. Database bytes are unchanged.")


## Interpretation boundaries

- MA and MI are a measured subset, not a national FanDuel growth rate. Revenue definitions differ by state.
- Ohio's two native FanDuel licensee rows include an old-license adjustment; aggregate them only after validating the transition. Kansas reports settled wagers and Net Revenues and needs its own rounding/definition contract.
- PA/NJ licensee totals cannot be assigned to FanDuel by a guessed brand mapping. The full exclusion table above remains part of the scorecard.
- Compare [New York weekly activity separately](94_flut_current_data.ipynb). Do not blend weeks and calendar months.
- This table can identify assumptions for review. State gross, adjusted or taxable revenue is not automatically Flutter's reported US net revenue.
